# GraphKV repaired simulator — Colab quick check
Run this top to bottom. The first run is deliberately small and diagnostic.

In [ ]:
from google.colab import files
uploaded = files.upload()  # choose graphkv_sim_repaired_v0.2.zip
zip_name = next(iter(uploaded))
!unzip -q -o "$zip_name" -d /content/

In [ ]:
%cd /content/graphkv_sim_repaired
!pip install -q -e ".[colab]"

## Gate 1: correctness tests
Do not proceed to paid GPU runs if any test fails.

In [ ]:
!pytest -q

## Gate 2: at most 20 events on a small model configuration
This downloads LongBench, a tokenizer/config, and MiniLM embeddings. It does not load Qwen causal-LM weights.

In [ ]:
!python run_simulation.py --smoke --output-dir outputs/colab_smoke

In [ ]:
!python validate_outputs.py outputs/colab_smoke

In [ ]:
import pandas as pd
summary = pd.read_csv('outputs/colab_smoke/summary.csv')
display(summary)
assert summary['events'].min() > 0
assert summary[['prediction_recall','residency_hit_rate']].apply(lambda s: s.between(0,1).all()).all()
coverage = summary['candidate_coverage'].dropna()
assert coverage.between(0,1).all()

## Inspect event semantics
Prediction misses and infrastructure misses must remain distinguishable.

In [ ]:
from pathlib import Path
files_ = sorted(Path('outputs/colab_smoke').glob('results_*.csv'))
events = pd.concat([pd.read_csv(p) for p in files_], ignore_index=True)
display(events.groupby(['policy','miss_reason']).size().rename('count').reset_index())
display(events[events.policy.str.contains('online')][['policy','event_id','prediction_hit','candidate_coverage','online_updated','online_update_reason','weights_before','weights_after']].head(30))

## Gate 3: stress smoke
A two-entry cache plus deadline mode exercises eviction and late-prefetch paths.

In [ ]:
!python run_simulation.py --stress-smoke --output-dir outputs/colab_stress
!python validate_outputs.py outputs/colab_stress

## Optional full historical-shape comparison
This runs 600 held-out test events for nine K values at capacity 16, with exactly 150 semantic, 150 structural, and 300 multi-hop queries. It remains a synthetic comparison diagnostic.

In [ ]:
# !python run_simulation.py --full-comparison-600 --model Qwen/Qwen2.5-1.5B-Instruct --dataset hotpot --output-dir outputs/full_600_qwen1_5b_hotpot
# !python validate_outputs.py outputs/full_600_qwen1_5b_hotpot

## Optional GPU-specific calibration
This loads the 0.5B causal LM. It is a microbenchmark, not an end-to-end serving result.

In [ ]:
# !python calibrate_small_model.py --model Qwen/Qwen2.5-0.5B-Instruct --output calibration.json